<a href="https://colab.research.google.com/github/nich02/AccountFraud/blob/main/RainfallDataExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Import necessary libraries
import ee, geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import ee
# Authenticate and initialize the Earth Engine API
ee.Authenticate()
ee.Initialize(project='cloudprojectname')


In [52]:
import geopandas

# Define the bounding box coordinates for Kenya and South Africa
kenya_coords = [(34.0, -5.0), (34.0, 5.0), (42.0, 5.0), (42.0, -5.0), (34.0, -5.0)]
south_africa_coords = [(16.0, -35.0), (16.0, -22.0), (33.0, -22.0), (33.0, -35.0), (16.0, -35.0)]

# Create GeoDataFrame for Kenya
kenya_gdf = gpd.GeoDataFrame([1], geometry=[geopandas.io.wkb.loads(ee.Geometry.Polygon(kenya_coords).toWkt().getInfo())], crs='EPSG:4326')

# Create GeoDataFrame for South Africa
south_africa_gdf = gpd.GeoDataFrame([1], geometry=[geopandas.io.wkb.loads(ee.Geometry.Polygon(south_africa_coords).toWkt().getInfo())], crs='EPSG:4326')

# Display the GeoDataFrames
display(kenya_gdf)
display(south_africa_gdf)

AttributeError: module 'geopandas.io' has no attribute 'wkb'

New Project

In [53]:
from shapely.geometry import Polygon

# Define the bounding box coordinates for Kenya and South Africa
kenya_coords = [(34.0, -5.0), (34.0, 5.0), (42.0, 5.0), (42.0, -5.0), (34.0, -5.0)]
south_africa_coords = [(16.0, -35.0), (16.0, -22.0), (33.0, -22.0), (33.0, -35.0), (16.0, -35.0)]

# Create GeoDataFrame for Kenya
kenya_gdf = gpd.GeoDataFrame([1], geometry=[Polygon(kenya_coords)], crs='EPSG:4326')

# Create GeoDataFrame for South Africa
south_africa_gdf = gpd.GeoDataFrame([1], geometry=[Polygon(south_africa_coords)], crs='EPSG:4326')

# Display the GeoDataFrames
display(kenya_gdf)
display(south_africa_gdf)

,0,geometry
0,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"


,0,geometry
0,1,"POLYGON ((16 -35, 16 -22, 33 -22, 33 -35, 16 -..."


In [54]:

# Define the time range for rainfall data extraction
start_date = '2023-01-01'
end_date = '2023-12-31'

# Function to extract rainfall data for a given region
def get_rainfall_data(roi, region_name):
    # Load the CHIRPS dataset for the specified date range
    chirps_dataset = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
        .filterDate(start_date, end_date) \
        .filterBounds(roi)

    # Calculate monthly total rainfall
    monthly_rainfall = chirps_dataset.map(lambda img: img.set(
        'month', ee.Date(img.get('system:time_start')).get('month')))

    # Define function to extract monthly rainfall statistics
    def extract_monthly_stats(image):
        stats = image.reduceRegion(
            reducer=ee.Reducer.minMax().combine(reducer2=ee.Reducer.mean(), sharedInputs=True),
            geometry=roi,
            scale=5000,  # Adjust scale based on your ROI size
            maxPixels=1e13
        )
        return stats

    # Create a list of monthly images and extract rainfall values
    monthly_values = []
    months = ee.List.sequence(1, 12)

    for month in months.getInfo():
        monthly_img = monthly_rainfall.filter(ee.Filter.eq('month', month)).sum()
        stats = extract_monthly_stats(monthly_img).getInfo()
        month_data = {'Month': month,
                      'Max_Rainfall_mm': stats.get('precipitation_max'),
                      'Avg_Rainfall_mm': stats.get('precipitation_mean'),
                      'Region': region_name}
        monthly_values.append(month_data)

    return monthly_values

# # Get rainfall data for Kenya and South Africa
# kenya_data = get_rainfall_data(kenya, 'Kenya')
# south_africa_data = get_rainfall_data(south_africa, 'South Africa')

# # Combine data and convert to DataFrame
# all_data = kenya_data + south_africa_data
# rainfall_df = pd.DataFrame(all_data)

# # Print the rainfall data as a formatted table
# print(rainfall_df.to_string(index=False))

In [55]:
# Add a 'Region' column to the GeoDataFrames for merging
kenya_gdf['Region'] = 'Kenya'
south_africa_gdf['Region'] = 'South Africa'

# Merge the rainfall DataFrame with the Kenya GeoDataFrame
kenya_rainfall_gdf = kenya_gdf.merge(rainfall_df, on='Region')

# Merge the rainfall DataFrame with the South Africa GeoDataFrame
south_africa_rainfall_gdf = south_africa_gdf.merge(rainfall_df, on='Region')

# Combine the merged GeoDataFrames
merged_rainfall_gdf = pd.concat([kenya_rainfall_gdf, south_africa_rainfall_gdf], ignore_index=True)

# Display the combined GeoDataFrame
display(merged_rainfall_gdf)

,0,geometry,Region,Month,Max_Rainfall_mm,Avg_Rainfall_mm
0,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,1,172.090835,15.935049
1,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,2,99.651565,8.874163
2,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,3,429.300228,106.738026
3,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,4,1158.992817,196.840379
4,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,5,426.950321,51.338490
5,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,6,414.060062,27.580381
6,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,7,293.747222,20.718296
7,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,8,237.838078,19.094835
8,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,9,353.385635,28.069954
9,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,10,573.234764,94.618802


## Export data into CSV

In [57]:
# Export the above into a CSV file
merged_rainfall_gdf.to_csv('Rainfall_SA_KN.csv', index=False)

###Let's try to read the exported CSV data

In [62]:
import pandas as pd
# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/Rainfall_SA_KN.csv')
# Display the first few rows of the DataFrame
df.head()

,0,geometry,Region,Month,Max_Rainfall_mm,Avg_Rainfall_mm
0,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,1,172.090835,15.935049
1,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,2,99.651565,8.874163
2,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,3,429.300228,106.738026
3,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,4,1158.992817,196.840379
4,1,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))",Kenya,5,426.950321,51.338490


## Export data into GeoJson

In [65]:
# Export data into GeoJson
import json
# Convert the GeoDataFrame to GeoJSON format
geojson_data = merged_rainfall_gdf.to_json()

# To save to a file:
with open('RainfallGeoJson_SA_KN.geojson', 'w') as f:
    json.dump(json.loads(geojson_data), f)

# # Print the GeoJSON data
# print(geojson_data)

### Let's try to read the exported Geojson

In [67]:
import geopandas as gpd

# Replace 'path/to/your/file.geojson' with the actual path to your GeoJSON file
gdf = gpd.read_file('/content/RainfallGeoJson_SA_KN.geojson')

gdf.head()
# gdf.geometry.head()

,id,0,Region,Month,Max_Rainfall_mm,Avg_Rainfall_mm,geometry
0,0,1,Kenya,1,172.090835,15.935049,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"
1,1,1,Kenya,2,99.651565,8.874163,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"
2,2,1,Kenya,3,429.300228,106.738026,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"
3,3,1,Kenya,4,1158.992817,196.840379,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"
4,4,1,Kenya,5,426.950321,51.338490,"POLYGON ((34 -5, 34 5, 42 5, 42 -5, 34 -5))"


## Export data into parquet format

In [68]:
# Now convert the dataframe into parquet
import pyarrow as pa
import pyarrow.parquet as pq
# Ensure all column names are strings
merged_rainfall_gdf.columns = merged_rainfall_gdf.columns.map(str)
# Now write to parquet
merged_rainfall_gdf.to_parquet('Rainfallparquet_SA_KN.parquet', index=False)

### Now Let's read the exported parquet file

In [51]:
import pandas as pd
df = pd.read_parquet('Rainfallparquet_SA_KN.parquet')
df.head()


,0,geometry,Region,Month,Max_Rainfall_mm,Avg_Rainfall_mm
0,1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,Kenya,1,172.090835,15.935049
1,1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,Kenya,2,99.651565,8.874163
2,1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,Kenya,3,429.300228,106.738026
3,1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,Kenya,4,1158.992817,196.840379
4,1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,Kenya,5,426.950321,51.338490
